In [1]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Cari file .env mulai dari folder kerja notebook, naik ke folder induk
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError(
        "File .env tidak ditemukan. Buat .env di folder proyek (sql-ecommerce-analysis)."
    )
load_dotenv(env_path)

# Pastikan semua variabel terisi
required = ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(f"Variabel belum terisi di .env: {missing}")

print("Konfigurasi .env berhasil dibaca.")

Konfigurasi .env berhasil dibaca.


In [2]:
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(db_url)

# Tes koneksi: engine baru benar-benar terhubung saat query pertama dijalankan
with engine.connect() as conn:
    info = conn.execute(text("SELECT current_database(), current_user")).fetchone()
print(f"Terhubung ke database '{info[0]}' sebagai user '{info[1]}'")

Terhubung ke database 'itversity_retail_db' sebagai user 'itversity_retail_user'


In [3]:
orders = pd.read_sql("SELECT * FROM orders", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
products = pd.read_sql("SELECT * FROM products", engine)
categories = pd.read_sql("SELECT * FROM categories", engine)
departments = pd.read_sql("SELECT * FROM departments", engine)
customers = pd.read_sql("SELECT * FROM customers", engine)

In [4]:
tables = {
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "categories": categories,
    "departments": departments,
    "customers": customers,
}

for name, df in tables.items():
    print(f"{name:<12} {df.shape[0]:>9,} baris x {df.shape[1]} kolom")

orders          68,883 baris x 4 kolom
order_items    172,198 baris x 6 kolom
products         1,345 baris x 6 kolom
categories          58 baris x 3 kolom
departments          6 baris x 2 kolom
customers       12,435 baris x 9 kolom


In [5]:
orders.head()

,order_id,order_date,order_customer_id,order_status
0,1,2013-07-25,11599,CLOSED
1,2,2013-07-25,256,PENDING_PAYMENT
2,3,2013-07-25,12111,COMPLETE
3,4,2013-07-25,8827,CLOSED
4,5,2013-07-25,11318,COMPLETE


In [6]:
orders.dtypes

order_id                      int64
order_date           datetime64[us]
order_customer_id             int64
order_status                    str
dtype: object

In [7]:
orders.isna().sum()

order_id             0
order_date           0
order_customer_id    0
order_status         0
dtype: int64

In [8]:
valid_orders = orders[orders["order_status"].isin(["COMPLETE", "CLOSED"])]

valid_items = valid_orders.merge(
    order_items,
    left_on="order_id",
    right_on="order_item_order_id",
    how="inner",
)

print("Order valid                    :", len(valid_orders))                        # target 30455
print("Order valid yang punya item    :", valid_items["order_id"].nunique())      # target 25266
print("Total valid revenue (USD)      :", round(valid_items["order_item_subtotal"].sum(), 2))  # target 15012982.48

Order valid                    : 30455
Order valid yang punya item    : 25266
Total valid revenue (USD)      : 15012982.48
